# SageMaker AI MLflow - DPO Fine-Tuning with Preference Data from Traces

In this notebook, you will build a **preference dataset** from MLflow traces and use **Direct Preference Optimization (DPO)** to fine-tune Qwen2.5-7B. DPO trains the model to prefer high-quality responses over low-quality ones without needing a separate reward model.

**What you will learn:**
- Build preference pairs (chosen/rejected) from MLflow traces using 3 approaches
- Run serverless DPO fine-tuning with DPOTrainer (LoRA) on Qwen2.5-7B Instruct
- Deploy and compare the DPO-trained model against chosen/rejected examples

### How We Build Preference Pairs

We pull traces from all three MLflow experiments and build preference pairs using different approaches:

| Source Experiment | Chosen | Rejected |
|---|---|---|
| medical-triage-agent | Thumbs-up traces (human feedback) | Thumbs-down traces (human feedback) |
| medical-triage-datacapture | follows_objective pass | follows_objective fail |
| medical-triage-offline-eval | High-quality traces | Base model re-generation (weaker prompt) |

### Prerequisites
- Completed the agent tracing lab (04-1) and online evaluation (04-2)
- Completed the evaluation labs with traces in all 3 experiments
- A SageMaker AI MLflow App
- Service quota for serverless fine-tuning and ml.g5.2xlarge for deployment


<div style="padding: 15px; background-color: #fff3cd; border-left: 5px solid #ffc107; color: #856404;">
<strong>Warning:</strong> The cell below installs libraries and restarts the kernel. After the restart, continue with the next cell.
</div>

In [ ]:
# Install SageMaker SDK v3 and dependencies
%pip install "sagemaker>=3.5.0" mlflow==3.9.0 sagemaker-mlflow==0.2.0 datasets==4.5.0 scikit-learn --quiet

# restart kernel
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

## 1. Environment Setup

In [ ]:
import json
import os
import boto3
import mlflow
import pandas as pd
from sagemaker.core.helper.session_helper import Session, get_execution_role
from utils.trace_utils import extract_prompt_response

sess = Session()
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix
region = sess.boto_region_name
s3_client = boto3.client("s3")

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

%store -r

print(f"Role: {role}")
print(f"Bucket: {bucket_name}")
print(f"Region: {region}")

## 2. Configure SageMaker AI MLflow

In [ ]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

agent_experiment_name = "medical-triage-agent"
datacapture_experiment_name = "medical-triage-datacapture"
offline_eval_experiment_name = "medical-triage-offline-eval"

dpo_experiment_name = "medical-triage-dpo"
mlflow.set_experiment(dpo_experiment_name)

print(f"MLflow tracking server: {mlflow.get_tracking_uri()}")
print(f"DPO experiment: {dpo_experiment_name}")

## 3. Human Feedback Preference Pairs (medical-triage-agent)

Pull traces from the agent experiment and use human feedback (thumbs up/down) to build preference pairs.
Traces with positive feedback become **chosen**, traces with negative feedback become **rejected**.

In [ ]:
mlflow.set_experiment(agent_experiment_name)
agent_traces = mlflow.search_traces()
print(f"Total traces in {agent_experiment_name}: {len(agent_traces)}")

thumbs_up_traces = []
thumbs_down_traces = []
no_feedback = 0

for idx, trace in agent_traces.iterrows():
    assessments = trace.get("assessments", [])
    if not assessments:
        no_feedback += 1
        continue
    has_feedback = False
    for a in assessments:
        name = a.get("assessment_name", "").lower()
        if "human" in name or "feedback" in name or "thumbs" in name:
            feedback = a.get("feedback", {})
            value = str(feedback.get("value", "")).lower()
            if value in ["yes", "thumbs_up", "1", "true", "good"]:
                thumbs_up_traces.append(trace)
                has_feedback = True
            elif value in ["no", "thumbs_down", "0", "false", "bad"]:
                thumbs_down_traces.append(trace)
                has_feedback = True
    if not has_feedback:
        no_feedback += 1

print(f"Traces with thumbs UP:   {len(thumbs_up_traces)}")
print(f"Traces with thumbs DOWN: {len(thumbs_down_traces)}")
print(f"Traces without feedback:  {no_feedback}")

<div style="padding: 15px; background-color: #d1ecf1; border-left: 5px solid #17a2b8; color: #0c5460;">
<strong>Tip:</strong> If you have not yet provided feedback on all traces, go to the MLflow UI, open the <code>medical-triage-agent</code> experiment, and add thumbs up or thumbs down to each trace. Then re-run the cell above.
</div>

In [ ]:
dpo_pairs_hf = []

for up_trace in thumbs_up_traces:
    up_prompt, up_response = extract_prompt_response(up_trace)
    if not up_prompt or not up_response or len(up_response.strip()) < 20:
        continue
    for down_trace in thumbs_down_traces:
        down_prompt, down_response = extract_prompt_response(down_trace)
        if not down_prompt or not down_response or len(down_response.strip()) < 20:
            continue
        dpo_pairs_hf.append({
            "prompt": up_prompt,
            "chosen": up_response,
            "rejected": down_response
        })

print(f"Human Feedback preference pairs: {len(dpo_pairs_hf)}")

## 4. Score-Based Preference Pairs (medical-triage-datacapture)

Pull traces from the DataCapture pipeline experiment.
Traces where follows_objective **passed** become **chosen**.
Traces where follows_objective **failed** become **rejected**.

In [ ]:
mlflow.set_experiment(datacapture_experiment_name)
dc_traces = mlflow.search_traces()
print(f"Total traces in {datacapture_experiment_name}: {len(dc_traces)}")

dc_passed = []
dc_failed = []

for idx, trace in dc_traces.iterrows():
    assessments = trace.get("assessments", [])
    if not assessments:
        continue
    for a in assessments:
        if "follows_objective" in a.get("assessment_name", "").lower():
            feedback = a.get("feedback", {})
            value = str(feedback.get("value", "")).lower()
            prompt, response = extract_prompt_response(trace)
            if not prompt or not response or len(response.strip()) < 20:
                break
            if value in ["no", "false", "fail", "0"]:
                dc_failed.append({"prompt": prompt, "response": response})
            else:
                dc_passed.append({"prompt": prompt, "response": response})
            break

print(f"Traces with follows_objective PASS: {len(dc_passed)}")
print(f"Traces with follows_objective FAIL: {len(dc_failed)}")

In [ ]:
dpo_pairs_sb = []

for chosen in dc_passed:
    for rejected in dc_failed:
        dpo_pairs_sb.append({
            "prompt": chosen["prompt"],
            "chosen": chosen["response"],
            "rejected": rejected["response"]
        })

print(f"Score-Based preference pairs: {len(dpo_pairs_sb)}")

## 5. LLM-Generated Rejected Responses (medical-triage-offline-eval)

Pull high-quality traces from the offline evaluation experiment as **chosen** responses.
Generate **rejected** responses by calling the base model (Qwen3-8B endpoint) with a deliberately weaker prompt (no system context).

In [ ]:
mlflow.set_experiment(offline_eval_experiment_name)
offline_traces = mlflow.search_traces()
print(f"Total traces in {offline_eval_experiment_name}: {len(offline_traces)}")

high_quality_traces = []
for idx, trace in offline_traces.iterrows():
    prompt, response = extract_prompt_response(trace)
    if not prompt or not response or len(response.strip()) < 20:
        continue
    high_quality_traces.append({"prompt": prompt, "response": response})

print(f"High-quality traces for LLM-generated pairs: {len(high_quality_traces)}")

In [ ]:
SAGEMAKER_ENDPOINT_NAME = "workshop-qwen3-8b"
sagemaker_runtime = boto3.client("sagemaker-runtime")

def generate_weak_response(prompt):
    """Generate a weaker response by calling the model without medical context."""
    weak_prompt = f"Answer briefly: {prompt}"
    request_body = {
        "messages": [{"role": "user", "content": weak_prompt}],
        "max_tokens": 256,
        "temperature": 1.0,
        "top_p": 0.5,
    }
    try:
        response = sagemaker_runtime.invoke_endpoint(
            EndpointName=SAGEMAKER_ENDPOINT_NAME,
            Body=json.dumps(request_body),
            ContentType="application/json",
        )
        result = json.loads(response["Body"].read().decode("utf-8"))
        return result["choices"][0]["message"]["content"]
    except Exception as e:
        print(f"Error generating weak response: {e}")
        return None

dpo_pairs_llm = []

print(f"Generating {len(high_quality_traces)} weak responses...")
for i, item in enumerate(high_quality_traces):
    rejected = generate_weak_response(item["prompt"])
    if rejected and len(rejected.strip()) > 10:
        dpo_pairs_llm.append({
            "prompt": item["prompt"],
            "chosen": item["response"],
            "rejected": rejected
        })
    if (i + 1) % 5 == 0:
        print(f"  Generated {i + 1}/{len(high_quality_traces)}...")

print(f"LLM-Generated preference pairs: {len(dpo_pairs_llm)}")

## 6. Combine All Preference Pairs

In [ ]:
all_dpo_pairs = dpo_pairs_hf + dpo_pairs_sb + dpo_pairs_llm

print(f"Total DPO preference pairs: {len(all_dpo_pairs)}")
print(f"  Human Feedback:   {len(dpo_pairs_hf)}")
print(f"  Score-Based:      {len(dpo_pairs_sb)}")
print(f"  LLM-Generated:    {len(dpo_pairs_llm)}")

if all_dpo_pairs:
    sample = all_dpo_pairs[0]
    print(f"\nSample pair:")
    print(f"  Prompt:   {sample['prompt'][:150]}...")
    print(f"  Chosen:   {sample['chosen'][:150]}...")
    print(f"  Rejected: {sample['rejected'][:150]}...")

## 7. Prepare DPO Dataset

Convert the preference pairs into JSONL format with prompt, chosen, rejected fields.
Split into training (80%) and validation (20%) sets, upload to S3, and register in SageMaker AI Registry.

In [ ]:
from sklearn.model_selection import train_test_split

train_records, val_records = train_test_split(all_dpo_pairs, test_size=0.2, random_state=42)

print(f"Training samples: {len(train_records)}")
print(f"Validation samples: {len(val_records)}")

In [ ]:
os.makedirs("./dpo_data/train", exist_ok=True)
os.makedirs("./dpo_data/val", exist_ok=True)

def save_jsonl(records, filepath):
    with open(filepath, "w") as f:
        for record in records:
            f.write(json.dumps(record) + "\n")

save_jsonl(train_records, "./dpo_data/train/dataset.jsonl")
save_jsonl(val_records, "./dpo_data/val/dataset.jsonl")

print(f"Train: {len(train_records)} records")
print(f"Val: {len(val_records)} records")

In [ ]:
if default_prefix:
    s3_dataset_prefix = f"{default_prefix}/datasets/medical-triage-dpo"
else:
    s3_dataset_prefix = "datasets/medical-triage-dpo"

dpo_train_s3_path = f"s3://{bucket_name}/{s3_dataset_prefix}/train/dataset.jsonl"
dpo_val_s3_path = f"s3://{bucket_name}/{s3_dataset_prefix}/val/dataset.jsonl"

s3_client.upload_file("./dpo_data/train/dataset.jsonl", bucket_name, f"{s3_dataset_prefix}/train/dataset.jsonl")
s3_client.upload_file("./dpo_data/val/dataset.jsonl", bucket_name, f"{s3_dataset_prefix}/val/dataset.jsonl")

import shutil
shutil.rmtree("./dpo_data")

print(f"Train: {dpo_train_s3_path}")
print(f"Val: {dpo_val_s3_path}")

## 8. Register Dataset in SageMaker AI Registry

In [ ]:
from sagemaker.ai_registry.dataset import DataSet
from sagemaker.ai_registry.dataset_utils import CustomizationTechnique

dataset_train = DataSet.create(
    name="medical-triage-dpo-train",
    source=dpo_train_s3_path,
    customization_technique=CustomizationTechnique.DPO,
    wait=True,
)
print(f"Training dataset ARN: {dataset_train.arn}")

dataset_val = DataSet.create(
    name="medical-triage-dpo-val",
    source=dpo_val_s3_path,
    customization_technique=CustomizationTechnique.DPO,
    wait=True,
)
print(f"Validation dataset ARN: {dataset_val.arn}")

In [ ]:
with mlflow.start_run(run_name="dpo-dataset-curation"):
    mlflow.log_param("source_experiments", f"{agent_experiment_name}, {datacapture_experiment_name}, {offline_eval_experiment_name}")
    mlflow.log_param("pairs_human_feedback", len(dpo_pairs_hf))
    mlflow.log_param("pairs_score_based", len(dpo_pairs_sb))
    mlflow.log_param("pairs_llm_generated", len(dpo_pairs_llm))
    mlflow.log_param("total_dpo_pairs", len(all_dpo_pairs))
    mlflow.log_param("train_samples", len(train_records))
    mlflow.log_param("val_samples", len(val_records))
    mlflow.log_param("train_s3_path", dpo_train_s3_path)
    mlflow.log_param("val_s3_path", dpo_val_s3_path)
    mlflow.set_tag("stage", "dpo-dataset-curation")

print("Dataset metadata logged to MLflow.")

## 9. DPO Fine-Tuning with DPOTrainer

We use the SageMaker DPOTrainer to run serverless DPO fine-tuning with LoRA on Qwen2.5-7B Instruct.
The trainer optimizes the model to prefer chosen responses over rejected ones.

In [ ]:
base_model_id = "huggingface-llm-qwen2-5-7b-instruct"

if default_prefix:
    dpo_output_path = f"s3://{bucket_name}/{default_prefix}/{base_model_id}-dpo"
else:
    dpo_output_path = f"s3://{bucket_name}/{base_model_id}-dpo"

os.environ["SAGEMAKER_MLFLOW_CUSTOM_ENDPOINT"] = f"https://mlflow.sagemaker.{region}.app.aws"

print(f"Base model: {base_model_id}")
print(f"Output path: {dpo_output_path}")

In [ ]:
from botocore.exceptions import ClientError
from sagemaker.core.resources import ModelPackageGroup

dpo_mpg_name = f"{base_model_id}-medical-triage-dpo-mpg"

try:
    mpg = ModelPackageGroup.get(model_package_group_name=dpo_mpg_name)
    print(f"Model Package Group exists: {dpo_mpg_name}")
except ClientError:
    mpg = ModelPackageGroup.create(
        model_package_group_name=dpo_mpg_name,
        model_package_group_description="Medical triage DPO models",
    )
    print(f"Created Model Package Group: {dpo_mpg_name}")

In [ ]:
from sagemaker.train.common import TrainingType
from sagemaker.train.dpo_trainer import DPOTrainer

trainer = DPOTrainer(
    model=base_model_id,
    training_type=TrainingType.LORA,
    model_package_group=dpo_mpg_name,
    training_dataset=dataset_train,
    validation_dataset=dataset_val,
    s3_output_path=dpo_output_path,
    sagemaker_session=sess,
    role=role,
    accept_eula=True,
)

trainer.hyperparameters.learning_rate = 0.0001
trainer.hyperparameters.global_batch_size = 16
trainer.hyperparameters.max_epochs = 1
trainer.hyperparameters.lr_warmup_ratio = 0.1

print("DPOTrainer configured. Hyperparameters:")
for k, v in trainer.hyperparameters.to_dict().items():
    print(f"  {k}: {v}")

<div style="padding: 15px; background-color: #d1ecf1; border-left: 5px solid #17a2b8; color: #0c5460;">
<strong>Note:</strong> The serverless DPO fine-tuning job typically takes 15-30 minutes. Training metrics are automatically tracked in MLflow.
</div>

In [ ]:
training_job = trainer.train(wait=True)

DPO_TRAINING_JOB_NAME = training_job.training_job_name
print(f"DPO Training job completed: {DPO_TRAINING_JOB_NAME}")

## 10. Deploy the DPO Fine-Tuned Model

Deploy the DPO-trained Qwen2.5-7B model using DJL LMI with vLLM on ml.g5.2xlarge.

<div style="padding: 15px; background-color: #f8d7da; border-left: 5px solid #dc3545; color: #721c24;">
<strong>Important:</strong> You only have 2x ml.g5.2xlarge endpoint instances available. If the SFT endpoint (workshop-qwen25-7b-sft) is still running, you must delete it before deploying the DPO model. Run the cell below to check.
</div>

In [ ]:
# Check if SFT endpoint exists and warn user
sm_client = boto3.client("sagemaker")
sft_endpoint_name = "workshop-qwen25-7b-sft"

try:
    response = sm_client.describe_endpoint(EndpointName=sft_endpoint_name)
    status = response["EndpointStatus"]
    print(f"WARNING: SFT endpoint '{sft_endpoint_name}' is {status}.")
    print(f"You must delete it before deploying the DPO model.")
    print(f"Uncomment and run the next cell to delete it.")
except sm_client.exceptions.ResourceNotFound:
    print(f"SFT endpoint '{sft_endpoint_name}' not found. Proceeding with DPO deployment.")
except Exception as e:
    if "Could not find" in str(e) or "does not exist" in str(e):
        print(f"SFT endpoint '{sft_endpoint_name}' not found. Proceeding with DPO deployment.")
    else:
        raise

In [ ]:
# Uncomment and run ONLY if the SFT endpoint needs to be deleted
# from sagemaker.core.resources import Endpoint
# Endpoint.get(endpoint_name="workshop-qwen25-7b-sft").delete()
# print("SFT endpoint deleted.")

### Get the Merged Model Artifact

In [ ]:
from sagemaker.core.resources import ModelPackage, ModelPackageGroup
from sagemaker.core import s3

model_package_version = "1"

mpg = ModelPackageGroup.get(dpo_mpg_name)
dpo_model_package_arn = (
    f"{mpg.model_package_group_arn.replace('model-package-group', 'model-package', 1)}/{model_package_version}"
)
print(f"Model Package ARN: {dpo_model_package_arn}")

model_package = ModelPackage.get(dpo_model_package_arn)

merged_model_s3_uri = (
    s3.s3_path_join(
        model_package.inference_specification.containers[0].model_data_source.s3_data_source.s3_uri,
        "checkpoints",
        "hf_merged",
    ) + "/"
)
print(f"Merged model S3 URI: {merged_model_s3_uri}")

### Create Endpoint and Deploy

In [ ]:
dpo_model_name = f"{base_model_id}-medical-triage-dpo"
dpo_endpoint_config_name = f"{base_model_id}-medical-triage-dpo-config"
dpo_endpoint_name = "workshop-qwen25-7b-dpo"

instance_type = "ml.g5.2xlarge"
health_check_timeout = 700

CONTAINER_VERSION = "0.36.0-lmi18.0.0-cu128"
inference_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/djl-inference:{CONTAINER_VERSION}"

env = {
    "HF_MODEL_ID": "/opt/ml/model",
    "OPTION_TRUST_REMOTE_CODE": "true",
    "OPTION_MODEL_LOADING_TIMEOUT": "3600",
    "OPTION_TENSOR_PARALLEL_DEGREE": "max",
    "SERVING_FAIL_FAST": "true",
    "OPTION_ROLLING_BATCH": "disable",
    "OPTION_ASYNC_MODE": "true",
    "OPTION_ENTRYPOINT": "djl_python.lmi_vllm.vllm_async_service",
    "OPTION_DTYPE": "bf16",
    "OPTION_QUANTIZE": "fp8",
    "OPTION_MAX_MODEL_LEN": json.dumps(1024 * 8),
}

print(f"Endpoint: {dpo_endpoint_name}")
print(f"Instance: {instance_type}")

In [ ]:
from sagemaker.core.resources import Model, Endpoint, EndpointConfig
from sagemaker.core.shapes import ContainerDefinition, ModelDataSource, S3ModelDataSource, ProductionVariant

fine_tuned_model = Model.create(
    model_name=dpo_model_name,
    primary_container=ContainerDefinition(
        image=inference_image,
        model_data_source=ModelDataSource(
            s3_data_source=S3ModelDataSource(
                s3_uri=merged_model_s3_uri,
                s3_data_type="S3Prefix",
                compression_type="None",
            )
        ),
        environment=env,
    ),
    execution_role_arn=role,
)
print(f"Model created: {dpo_model_name}")

endpoint_config = EndpointConfig.create(
    endpoint_config_name=dpo_endpoint_config_name,
    production_variants=[
        ProductionVariant(
            variant_name="AllTraffic",
            model_name=dpo_model_name,
            instance_type=instance_type,
            initial_instance_count=1,
            model_data_download_timeout_in_seconds=health_check_timeout,
        )
    ],
)
print(f"EndpointConfig created: {dpo_endpoint_config_name}")

endpoint = Endpoint.create(
    endpoint_name=dpo_endpoint_name, endpoint_config_name=dpo_endpoint_config_name
)
endpoint.wait_for_status("InService")
print(f"Endpoint {dpo_endpoint_name} is InService!")

## 11. Test: Compare DPO Model with Chosen vs Rejected

Compare the DPO model output against the chosen (high-quality) and rejected (weak base model) responses from the training data.

In [ ]:
import io

sagemaker_runtime = boto3.client("sagemaker-runtime")
TEACHER_ENDPOINT = "workshop-qwen3-8b"

class LineIterator:
    def __init__(self, stream):
        self.byte_iterator = iter(stream)
        self.buffer = io.BytesIO()
        self.read_pos = 0
    def __iter__(self):
        return self
    def __next__(self):
        while True:
            self.buffer.seek(self.read_pos)
            line = self.buffer.readline()
            if line and line[-1] == ord("\n"):
                self.read_pos += len(line)
                return line[:-1]
            try:
                chunk = next(self.byte_iterator)
            except StopIteration:
                if self.read_pos < self.buffer.getbuffer().nbytes:
                    continue
                raise
            if "PayloadPart" not in chunk:
                continue
            self.buffer.seek(0, io.SEEK_END)
            self.buffer.write(chunk["PayloadPart"]["Bytes"])

def invoke_dpo_model(prompt):
    """Invoke the DPO fine-tuned model with streaming."""
    request_body = {
        "messages": [{"role": "user", "content": [{"type": "text", "text": prompt}]}],
        "max_tokens": 1024,
        "temperature": 0.3,
        "top_p": 0.9,
        "stop": ["<|im_end|>"],
        "stream": True,
    }
    response = sagemaker_runtime.invoke_endpoint_with_response_stream(
        EndpointName=dpo_endpoint_name,
        Body=json.dumps(request_body),
        ContentType="application/json",
    )
    generated_text = ""
    for line in LineIterator(response["Body"]):
        if line:
            line_str = line.decode("utf-8")
            if not line_str.strip() or line_str.strip() == "data: [DONE]":
                continue
            if line_str.startswith("data: "):
                line_str = line_str[6:]
            try:
                data = json.loads(line_str)
                if "choices" in data:
                    for choice in data["choices"]:
                        if "delta" in choice and "content" in choice["delta"]:
                            content = choice["delta"]["content"]
                            generated_text += content
                            print(content, end="", flush=True)
            except json.JSONDecodeError:
                pass
    print()
    return generated_text

def invoke_teacher_model(prompt):
    """Invoke the teacher model (Qwen3-8B) for comparison."""
    request_body = {
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 1024,
        "temperature": 0.3,
    }
    response = sagemaker_runtime.invoke_endpoint(
        EndpointName=TEACHER_ENDPOINT,
        Body=json.dumps(request_body),
        ContentType="application/json",
    )
    result = json.loads(response["Body"].read().decode("utf-8"))
    return result["choices"][0]["message"]["content"]

print("Inference utilities ready.")

### Test 1: DPO Model vs Chosen vs Rejected

Compare the DPO model output against the chosen (high-quality) and rejected (weak) responses from the LLM-generated preference pairs.

In [ ]:
test_pair = dpo_pairs_llm[0]

print("=" * 60)
print("PROMPT:")
print("=" * 60)
print(test_pair["prompt"])
print()

print("=" * 60)
print("DPO MODEL RESPONSE:")
print("=" * 60)
dpo_response = invoke_dpo_model(test_pair["prompt"])
print()

print("=" * 60)
print("CHOSEN (high-quality trace from offline evaluation):")
print("=" * 60)
print(test_pair["chosen"][:800])
print()

print("=" * 60)
print("REJECTED (weak response from base model):")
print("=" * 60)
print(test_pair["rejected"][:800])

### Test 2: DPO Model vs Teacher Model

Compare the DPO-trained 7B model against the teacher 8B model on the same medical triage query.

In [ ]:
test_prompt = (
    "Assess the following patient and provide the likely condition, "
    "triage level, and treatment protocol.\n\n"
    "Patient ID: PT-001\nAge: 62\nSymptoms: chest pain, shortness of breath, sweating\n"
    "Reported Severity: high"
)

print("=" * 60)
print("DPO MODEL RESPONSE (Qwen2.5-7B DPO-trained):")
print("=" * 60)
dpo_response = invoke_dpo_model(test_prompt)
print()

print("=" * 60)
print("TEACHER MODEL RESPONSE (Qwen3-8B base):")
print("=" * 60)
teacher_response = invoke_teacher_model(test_prompt)
print(teacher_response)

## 12. Cleanup

Uncomment and run the cells below to delete deployed resources.

In [ ]:
# Delete DPO endpoint
# Endpoint.get(endpoint_name=dpo_endpoint_name).delete()
# print(f"Deleted Endpoint: {dpo_endpoint_name}")

# Delete endpoint config
# EndpointConfig.get(endpoint_config_name=dpo_endpoint_config_name).delete()
# print(f"Deleted EndpointConfig: {dpo_endpoint_config_name}")

# Delete model
# Model.get(model_name=dpo_model_name).delete()
# print(f"Deleted Model: {dpo_model_name}")

In [ ]:
# Also delete the teacher model endpoint if no longer needed
# Endpoint.get(endpoint_name="workshop-qwen3-8b").delete()
# print("Deleted teacher endpoint: workshop-qwen3-8b")